In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm

from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA, QuadraticDiscriminantAnalysis as QDA
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay


from ISLP import load_data

## Stock Market Data

In [ ]:
smarket = load_data("Smarket")
smarket.columns = [c.lower() for c in smarket.columns]

In [ ]:
label = "direction"
all_features = [c for c in smarket.columns if c != label]

## Logistic Regression

In [ ]:
features = [f for f in all_features if f not in ["today", "year"]]

train_mask = smarket["year"] < 2005
train_df, valid_df = smarket.loc[train_mask], smarket.loc[~train_mask]
X_train, y_train = train_df[features], train_df[label]
X_valid, y_valid = valid_df[features], valid_df[label]

y_train = (y_train == "Up").astype(float)
y_valid = (y_valid == "Up").astype(float)

print(X_train.shape, y_train.shape)
print(X_valid.shape, y_valid.shape)

In [ ]:
glm = sm.GLM(y_train, X_train, family=sm.families.Binomial())
results = glm.fit()
results.summary()

In [ ]:
y_hat_probas = results.predict(exog=X_valid)
y_hat_cls = (y_hat_probas > 0.5).astype(float)

In [ ]:
conf_matrix = confusion_matrix(y_valid, y_hat_cls)
ConfusionMatrixDisplay(conf_matrix).plot()

In [ ]:
(y_hat_cls == y_valid).mean()

## Discriminant Analysis

In [ ]:
features = [f for f in all_features if f not in ["today", "year"]]

train_mask = smarket["year"] < 2005
train_df, valid_df = smarket.loc[train_mask], smarket.loc[~train_mask]
X_train, y_train = train_df[features], train_df[label]
X_valid, y_valid = valid_df[features], valid_df[label]

y_train = (y_train == "Up").astype(float)
y_valid = (y_valid == "Up").astype(float)

print(X_train.shape, y_train.shape)
print(X_valid.shape, y_valid.shape)

In [ ]:
lda = LDA(store_covariance=True)
lda = lda.fit(X_train, y_train)

qda = QDA(store_covariance=True)
qda = qda.fit(X_train, y_train)

In [ ]:
# LDA
y_hat_probas = lda.predict_proba(X=X_valid)[:, 1]
y_hat_cls = (y_hat_probas > 0.5).astype(float)
(y_hat_cls == y_valid).mean()

In [ ]:
# QDA
y_hat_probas = qda.predict_proba(X=X_valid)[:, 1]
y_hat_cls = (y_hat_probas > 0.5).astype(float)
(y_hat_cls == y_valid).mean()

## Logistic Regression vs. LDA vs. QDA

In [ ]:
n_samples = 200

mu_1 = np.array([-1.5, -2.0])
mu_2 = np.array([3.5, -1.5])
mu_3 = np.array([0.5, 2.5])

cov_1 = np.array([[1.0, 0.5],[0.5, 1.0]])
cov_2 = np.array([[1.0, 0.4],[0.4, 1.0]])
cov_3 = np.array([[1.0, -0.2],[-0.2, 1.0]])

samples_1 = np.random.multivariate_normal(mean=mu_1, cov=cov_1, size=n_samples)
samples_2 = np.random.multivariate_normal(mean=mu_2, cov=cov_2, size=n_samples)
samples_3 = np.random.multivariate_normal(mean=mu_3, cov=cov_3, size=n_samples)

In [ ]:
plt.figure()
plt.scatter(samples_1[:, 0], samples_1[:, 1], color="blue", marker="o", alpha=0.5, label="Class 1")
plt.scatter(samples_2[:, 0], samples_2[:, 1], color="red", marker="x", alpha=0.5, label="Class 2")
plt.scatter(samples_3[:, 0], samples_3[:, 1], color="green", marker="*", alpha=0.5, label="Class 3")

plt.legend()
plt.xlabel("X1")
plt.ylabel("X2");

In [ ]:
labels_1 = np.ones(shape=(n_samples, ))
labels_2 = np.ones(shape=(n_samples, )) * 2
labels_3 = np.ones(shape=(n_samples, )) * 3

X = np.vstack([samples_1, samples_2, samples_3])
y = np.concatenate([labels_1, labels_2, labels_3])

# Logistic Regression
lr = LogisticRegression()
lr = lr.fit(X=X, y=y)

# LDA
lda = LDA()
lda = lda.fit(X, y)

# QDA
qda = QDA()
qda = qda.fit(X, y)

In [ ]:
# Predict
x1s = np.linspace(start=-5, stop=6, num=100)
x2s = np.linspace(start=-5, stop=6, num=100)
x1_test, x2_test = np.meshgrid(x1s, x2s)
X_test = np.hstack([x1_test.reshape(-1, 1), x2_test.reshape(-1, 1)])

In [ ]:
# Logistic Regression
y_hat = lr.predict(X_test)

plt.figure()

plt.scatter(X_test[y_hat==1][:, 0], X_test[y_hat==1][:, 1], color="blue", marker="o", s=1, alpha=0.5)
plt.scatter(X_test[y_hat==2][:, 0], X_test[y_hat==2][:, 1], color="red", marker="o", s=1, alpha=0.5)
plt.scatter(X_test[y_hat==3][:, 0], X_test[y_hat==3][:, 1], color="green", marker="o", s=1, alpha=0.5)

plt.scatter(samples_1[:, 0], samples_1[:, 1], color="blue", marker="o", alpha=0.7)
plt.scatter(samples_2[:, 0], samples_2[:, 1], color="red", marker="x", alpha=0.7)
plt.scatter(samples_3[:, 0], samples_3[:, 1], color="green", marker="*", alpha=0.7)

plt.xlabel("X1")
plt.ylabel("X2");

In [ ]:
# Logistic Regression
y_hat = lda.predict(X_test)

plt.figure()

plt.scatter(X_test[y_hat==1][:, 0], X_test[y_hat==1][:, 1], color="blue", marker="o", s=1, alpha=0.5)
plt.scatter(X_test[y_hat==2][:, 0], X_test[y_hat==2][:, 1], color="red", marker="o", s=1, alpha=0.5)
plt.scatter(X_test[y_hat==3][:, 0], X_test[y_hat==3][:, 1], color="green", marker="o", s=1, alpha=0.5)

plt.scatter(samples_1[:, 0], samples_1[:, 1], color="blue", marker="o", alpha=0.7)
plt.scatter(samples_2[:, 0], samples_2[:, 1], color="red", marker="x", alpha=0.7)
plt.scatter(samples_3[:, 0], samples_3[:, 1], color="green", marker="*", alpha=0.7)

plt.xlabel("X1")
plt.ylabel("X2");

In [ ]:
# Logistic Regression
y_hat = qda.predict(X_test)

plt.figure()

plt.scatter(X_test[y_hat==1][:, 0], X_test[y_hat==1][:, 1], color="blue", marker="o", s=1, alpha=0.5)
plt.scatter(X_test[y_hat==2][:, 0], X_test[y_hat==2][:, 1], color="red", marker="o", s=1, alpha=0.5)
plt.scatter(X_test[y_hat==3][:, 0], X_test[y_hat==3][:, 1], color="green", marker="o", s=1, alpha=0.5)

plt.scatter(samples_1[:, 0], samples_1[:, 1], color="blue", marker="o", alpha=0.7)
plt.scatter(samples_2[:, 0], samples_2[:, 1], color="red", marker="x", alpha=0.7)
plt.scatter(samples_3[:, 0], samples_3[:, 1], color="green", marker="*", alpha=0.7)

plt.xlabel("X1")
plt.ylabel("X2");

## K-Nearest Neighbours

In [ ]:
caravan = load_data("Caravan")